# TradePulse: Final Model Training & Verification (Phase 0, 1, 2)
**Author:** TradePulse Engineering Team (IIT Patna, IMPACT Lab)  
**Target:** META (NASDAQ), 5-Minute Closed Bars, 15-Minute Forward Direction  
**Frozen Configuration:** `label_fixed_15bp`, 50% Random Forest + 50% Logistic Regression, Platt Sigmoid Calibrator  

This notebook documents and reproduces:  
1. **Phase 0:** Configuration freeze from selection artifacts.  
2. **Phase 1:** Recency weighting evaluation across predefined grid (uniform, 180d, 90d, 45d) via PurgedWalkForwardCV with paired day-block bootstrap.  
3. **Phase 2:** One-time held-out test evaluation on sealed Test Set (Sep 2 - Nov 6, 2026, 47 sessions).

In [ ]:
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path(".." if Path.cwd().name == "notebooks" else ".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ml.train_final import run_phase1_cv, run_phase2_test_evaluation
print("Project Root:", PROJECT_ROOT)

## Phase 0: Verify Frozen Configuration

In [ ]:
config_path = PROJECT_ROOT / "artifacts" / "metadata" / "final_config_final.json"
with open(config_path) as f:
    config = json.load(f)
print(json.dumps(config, indent=2))

## Phase 1: Recency Weighting Grid & Paired Bootstrap Results
Grid: Uniform (no decay), 180, 90, 45 trading days.

In [ ]:
p1_path = PROJECT_ROOT / "artifacts" / "metadata" / "phase1_recency_weighting_results.json"
with open(p1_path) as f:
    p1_results = json.load(f)

print("CHOSEN HALF-LIFE:", p1_results["chosen_half_life"])
print("RATIONALE:", p1_results["decision_rationale"])
print("\nGRID METRICS SUMMARY:")
grid_df = pd.DataFrame(p1_results["grid_results"]).T
display(grid_df[["ess", "ess_pct", "auc_updown", "sign_accuracy", "gross_return_bps", "net_return_8bp_bps"]])

In [ ]:
# Display Recency Weighting CV Figure
from IPython.display import Image, display
fig_p1 = PROJECT_ROOT / "reports" / "figures" / "recency_weighting_cv_final.png"
display(Image(filename=str(fig_p1)))

## Phase 2: One-Time Final Test Evaluation Results (Held-Out Sep 2 - Nov 6, 2026)

In [ ]:
p2_path = PROJECT_ROOT / "artifacts" / "metadata" / "phase2_test_results.json"
with open(p2_path) as f:
    p2_results = json.load(f)

print("Held-Out Test Results Summary:")
for k in ["trading_days", "valid_bars", "tau_conviction_threshold", "total_signals", "signals_per_day", "zero_signal_days_pct", "precision", "binary_sign_accuracy", "auc_updown", "gross_return_bps", "net_return_8bp_bps"]:
    print(f"  {k:30s}: {p2_results[k]}")

print("\nBaselines Comparison:")
display(pd.DataFrame(p2_results["baselines_comparison"]).T)

In [ ]:
# Display Test Figures
fig_eq = PROJECT_ROOT / "reports" / "figures" / "test_equity_curve_final.png"
fig_cal = PROJECT_ROOT / "reports" / "figures" / "test_calibration_curve_final.png"
display(Image(filename=str(fig_eq)))
display(Image(filename=str(fig_cal)))